# 🎬 Headless Media Automation Pipeline (Google Colab)
**100% Free Cloud Worker** for downloading media at 1 Gbps, uploading directly to Doodstream, and publishing to your live InfinityFree WordPress CMS.

- **Step 1:** Run **Cell 1** to install aria2, FFmpeg, and Python libraries.
- **Step 2:** Enter your live WordPress & Doodstream credentials in **Cell 2**.
- **Step 3:** Paste a Magnet link, direct video URL, or movie name in **Cell 4** to execute the pipeline!

In [ ]:
# @title ⚙️ 1. Install Dependencies & Cloud Environment
!apt-get update -qq
!apt-get install -y -qq aria2 ffmpeg
!pip install -q requests requests-toolbelt beautifulsoup4 python-dotenv tqdm
!mkdir -p /content/downloads
print("✅ System & Python dependencies installed successfully!")

In [ ]:
# @title 🔑 2. Cloud Configuration & Credentials
# @markdown Fill in your live InfinityFree WordPress and Doodstream credentials below:

DOODSTREAM_API_KEY = "578084xvwvf2mt7is4dgrb" # @param {type:"string"}
WP_SITE_URL = "https://your-domain.infinityfreeapp.com" # @param {type:"string"}
WP_USERNAME = "admin" # @param {type:"string"}
WP_APP_PASSWORD = "" # @param {type:"string"}
TMDB_API_KEY = "" # @param {type:"string"}

WP_SITE_URL = WP_SITE_URL.rstrip('/')
print(f"✅ Target WordPress CMS: {WP_SITE_URL}")

In [ ]:
# @title 🛠️ 3. Pipeline Core Functions (Aria2c + Doodstream + WordPress API)
import os, sys, re, json, time, requests, subprocess
from urllib.parse import quote
from requests.auth import HTTPBasicAuth
from requests_toolbelt import MultipartEncoder, MultipartEncoderMonitor

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
}

def clean_title(name):
    clean = re.sub(r'\[.*?\]|\(.*?\)', '', name)
    clean = re.sub(r'\b(1080p|720p|480p|2160p|4k|bluray|web-dl|webrip|hdrip|x264|x265|hevc|aac|egydead|yts|yify)\b', '', clean, flags=re.I)
    clean = clean.replace('.', ' ').replace('_', ' ').strip()
    ym = re.search(r'\b(19\d\d|20\d\d)\b', clean)
    year = ym.group(1) if ym else None
    if year: clean = re.sub(r'\b' + year + r'\b', '', clean).strip()
    return clean.strip(), year

def fetch_metadata(title, year=None):
    print(f"[META] Querying metadata for '{title}' ({year or 'Any'})...")
    # 1. TMDb
    if TMDB_API_KEY:
        try:
            url = f"https://api.themoviedb.org/3/search/movie?api_key={TMDB_API_KEY}&query={quote(title)}"
            if year: url += f"&year={year}"
            r = requests.get(url, headers=HEADERS, timeout=15).json()
            if r.get("results"):
                m = r["results"][0]
                p = m.get("poster_path")
                return {
                    "title": m.get("title", title),
                    "year": m.get("release_date", "")[:4] or year or "2026",
                    "summary": m.get("overview", "No synopsis available."),
                    "rating": str(round(m.get("vote_average", 7.5), 1)),
                    "genres": "Action, Adventure",
                    "cover_url": f"https://image.tmdb.org/t/p/w780{p}" if p else None
                }
        except Exception as e:
            print(f"[META] TMDb error: {e}")

    # 2. YTS Mirror fallback
    try:
        r = requests.get(f"https://yts.mx/api/v2/list_movies.json?query_term={quote(title)}", headers=HEADERS, timeout=15).json()
        if r.get("status") == "ok" and r.get("data", {}).get("movie_count", 0) > 0:
            m = r["data"]["movies"][0]
            return {
                "title": m.get("title", title),
                "year": str(m.get("year", year or "2026")),
                "summary": m.get("summary", "No synopsis available."),
                "rating": str(m.get("rating", "7.0")),
                "genres": ", ".join(m.get("genres", ["Movies"])),
                "cover_url": m.get("large_cover_image") or m.get("medium_cover_image")
            }
    except Exception as e:
        print(f"[META] YTS fallback error: {e}")

    return {
        "title": title, "year": year or "2026",
        "summary": f"Watch {title} online in full HD.",
        "rating": "7.5", "genres": "Action, Drama",
        "cover_url": "https://images.unsplash.com/photo-1489599849927-2ee91cede3ba?w=800&q=80"
    }

def download_with_aria2(source, dest_dir="/content/downloads"):
    os.makedirs(dest_dir, exist_ok=True)
    print(f"[ARIA2] Downloading: {source[:80]}...")
    cmd = ["aria2c", f"--dir={dest_dir}", "-s8", "-x8", "--seed-time=0", source]
    subprocess.run(cmd, check=True)
    videos = []
    for root, _, files in os.walk(dest_dir):
        for f in files:
            if f.lower().endswith((".mp4", ".mkv", ".avi", ".webm")):
                fp = os.path.join(root, f)
                videos.append((fp, os.path.getsize(fp)))
    if not videos:
        raise FileNotFoundError("No video file downloaded.")
    videos.sort(key=lambda x: x[1], reverse=True)
    video_path = videos[0][0]
    print(f"[ARIA2] Downloaded: {video_path} ({videos[0][1]/(1024*1024):.1f} MB)")
    return video_path

def upload_to_doodstream(video_path):
    print("[DOOD] Getting upload server...")
    srv = requests.get("https://doodapi.com/api/upload/server", params={"key": DOODSTREAM_API_KEY}, timeout=30).json()
    upload_url = srv["result"]
    fname = os.path.basename(video_path)
    print(f"[DOOD] Streaming upload for {fname}...")
    with open(video_path, 'rb') as f:
        encoder = MultipartEncoder(fields={'api_key': DOODSTREAM_API_KEY, 'file': (fname, f, 'video/mp4')})
        last = [-1]
        def progress(m):
            pct = int((m.bytes_read / m.len) * 100)
            if pct % 10 == 0 and pct != last[0]:
                last[0] = pct
                print(f"[DOOD Progress] {pct}% ({m.bytes_read/(1024*1024):.1f} MB)", flush=True)
        monitor = MultipartEncoderMonitor(encoder, progress)
        resp = requests.post(upload_url, data=monitor, headers={'Content-Type': monitor.content_type}, timeout=7200).json()
    code = resp["result"][0]["filecode"] if isinstance(resp["result"], list) else resp["result"]["filecode"]
    embed = f"https://dood.to/e/{code}"
    print(f"[DOOD] ✅ Upload complete: {embed}")
    return embed

def upload_poster(cover_url):
    if not cover_url: return None
    print(f"[WP] Fetching poster from {cover_url}...")
    img_data = requests.get(cover_url, headers=HEADERS, timeout=30).content
    h = {
        "Content-Disposition": f"attachment; filename=poster_{int(time.time())}.jpg",
        "Content-Type": "image/jpeg",
        "User-Agent": HEADERS["User-Agent"]
    }
    auth = HTTPBasicAuth(WP_USERNAME, WP_APP_PASSWORD) if WP_APP_PASSWORD else None
    res = requests.post(f"{WP_SITE_URL}/wp-json/wp/v2/media", headers=h, data=img_data, auth=auth, timeout=30)
    if res.status_code in (200, 201):
        mid = res.json()["id"]
        print(f"[WP] ✅ Poster uploaded (ID: {mid})")
        return mid
    print(f"[WP] Poster upload warning ({res.status_code}): {res.text[:100]}")
    return None

def publish_post(meta, embed_url, media_id):
    title = f"{meta['title']} ({meta['year']})"
    print(f"[WP] Publishing: {title}...")
    content = f"""<p><strong>Rating:</strong> {meta.get('rating', 'N/A')} / 10</p>
<p><strong>Genres:</strong> {meta.get('genres', 'Movies')}</p><hr>
<p>{meta.get('summary', '')}</p>
<div style='position:relative;padding-bottom:56.25%;height:0;overflow:hidden;'>
  <iframe src='{embed_url}' style='position:absolute;top:0;left:0;width:100%;height:100%;border:0;' allowfullscreen scrolling='no'></iframe>
</div>"""
    payload = {
        "title": title, "content": content, "status": "publish",
        "meta": {"video_year": str(meta.get("year", "2026")), "imdb_rating": str(meta.get("rating", "7.5"))}
    }
    if media_id: payload["featured_media"] = media_id
    auth = HTTPBasicAuth(WP_USERNAME, WP_APP_PASSWORD) if WP_APP_PASSWORD else None
    res = requests.post(f"{WP_SITE_URL}/wp-json/wp/v2/posts", json=payload, headers=HEADERS, auth=auth, timeout=30)
    if res.status_code in (200, 201):
        data = res.json()
        print(f"🎉 SUCCESS: Movie published to WordPress!")
        print(f"WordPress Post URL: {data.get('link')}")
        return data
    raise RuntimeError(f"WP publish failed: {res.status_code} {res.text[:200]}")

In [ ]:
# @title 🚀 4. Run Movie Pipeline
# @markdown Paste a Magnet link, Direct Video URL, or local path:
SOURCE_URL = "https://example.com/movie.mp4" # @param {type:"string"}
OVERRIDE_TITLE = "" # @param {type:"string"}
OVERRIDE_YEAR = "" # @param {type:"string"}

# 1. Download
if os.path.exists(SOURCE_URL):
    video_file = SOURCE_URL
    t, y = clean_title(os.path.basename(video_file))
else:
    video_file = download_with_aria2(SOURCE_URL)
    t, y = clean_title(os.path.basename(video_file))

movie_title = OVERRIDE_TITLE.strip() or t
movie_year = OVERRIDE_YEAR.strip() or y or "2026"

# 2. Metadata
meta = fetch_metadata(movie_title, movie_year)

# 3. Upload to Doodstream
embed_url = upload_to_doodstream(video_file)

# 4. Upload Poster to WordPress
media_id = upload_poster(meta.get("cover_url"))

# 5. Publish to WordPress
post_info = publish_post(meta, embed_url, media_id)
print(f"\n🍿 Done! Check your live Next.js Vercel frontend in 60 seconds to see the new movie live!")